In [1]:
import os
import sys
import time
import cv2
import mss
import numpy as np
import onnxruntime as ort
import pygetwindow as gw


In [2]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "../.."))
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
sys.path.append(SRC_DIR)

ONNX_MODEL_PATH = os.path.join(PROJECT_ROOT, "src", "resnet18", "checkpoints", "resnet18_terraria.onnx")

CLASSES = ['Corruption', 'Crimson', 'Desert', 'Dungeon', 'Forest', 'Hallow',
           'Hell', 'Jungle', 'Mushroom', 'Ocean', 'Snow', 'Space', 'Underground']


In [3]:
# MEAN and STD from training
MEAN = np.array([0.1473, 0.1647, 0.2079], dtype=np.float32)
STD  = np.array([0.1967, 0.2150, 0.2937], dtype=np.float32)
TARGET_W, TARGET_H = 384, 216

def prep_frame(frame):
    """ Convert BGR frame to model input """
    x = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x = cv2.resize(x, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
    x = (x.astype(np.float32) / 255.0 - MEAN) / STD
    return x.transpose(2, 0, 1)[None]  # 1x3xHxW

In [4]:
sess = ort.InferenceSession(ONNX_MODEL_PATH, providers=[
    p for p in ['CUDAExecutionProvider', 'CPUExecutionProvider'] 
    if p in ort.get_available_providers()
])
inp_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name


In [5]:

def get_terraria_monitor():
    try:
        window = gw.getWindowsWithTitle("Terraria")[0]
        left, top = window.left, window.top
        width, height = window.width, window.height
        return {'left': left, 'top': top, 'width': width, 'height': height}
    except IndexError:
        # fallback: full monitor
        return None

In [6]:
cv2.namedWindow('Biome', cv2.WINDOW_NORMAL)
with mss.mss() as sct:
    try:
        while True:
            monitor_region = get_terraria_monitor()
            if monitor_region is None:
                frame_full = np.array(sct.grab(sct.monitors[1]))[:, :, :3].copy()
            else:
                frame_full = np.array(sct.grab(sct.monitors[1]))[:, :, :3].copy()

                mon = sct.monitors[1]
                top_clip = max(0, monitor_region['top'])
                left_clip = max(0, monitor_region['left'])
                bottom_clip = min(mon['height'], monitor_region['top'] + monitor_region['height'])
                right_clip = min(mon['width'], monitor_region['left'] + monitor_region['width'])
                
                
                # Crop the Terraria window area
                frame_full = np.array(sct.grab(sct.monitors[1]))[:, :, :3].copy()
                frame = frame_full[top_clip:bottom_clip, left_clip:right_clip]

            h, w, _ = frame.shape
            crop_h, crop_w = 216, 384
            center_y, center_x = h // 2, w // 2
            top = max(center_y - crop_h // 2, 0)
            left = max(center_x - crop_w // 2, 0)
            frame_crop = frame[top:top+crop_h, left:left+crop_w]

            
            input_tensor = prep_frame(frame_crop)
            logits = sess.run([out_name], {inp_name: input_tensor})[0][0]

            # Convert logits to probabilities
            probs = np.exp(logits - logits.max())
            probs /= probs.sum()
            idx = probs.argmax()

            # Overlay main prediction on crop
            display_frame = frame_crop.copy()
            cv2.putText(display_frame, f'{CLASSES[idx]} ({probs[idx]:.3f})',
                        (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Optional: overlay top 3 predictions
            top3_idx = probs.argsort()[-3:][::-1]
            for i, t in enumerate(top3_idx):
                cv2.putText(display_frame, f"{CLASSES[t]}: {probs[t]:.2f}",
                            (20, 80 + i*30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

            cv2.imshow('Biome', display_frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            # Optional FPS limiter for 60 FPS
            time.sleep(0.03)

    finally:
        cv2.destroyAllWindows()